# Converting structures: CIF, ASE and DiffPy

This notebook demonstrates the interop adapters.  It requires the extras
(`pip install 'discus-python[interop]'`); without them the conversion cells
print a skip message and the notebook still runs.

See [docs/INTEROP.md](../docs/INTEROP.md) for the full documentation.

In [1]:
from pathlib import Path
import os
import sys
sys.path.insert(0, str(Path.cwd().parent))
os.chdir(Path.cwd().parent)

import numpy as np
from discuspdf import Lattice, Structure
from discuspdf.cif_io import read_cif, write_cif

FIXTURES = Path('tests/fixtures/interop')

## Read a real CIF (COD GaAs)

In [2]:
imp = read_cif(FIXTURES / 'gaas_cod_9008845.cif')
s = imp.structure
print(f'CIF: {imp.block}, space group {imp.spacegroup}, '
      f'{imp.n_asymmetric} asymmetric sites -> {s.n_atoms} atoms')
assert s.n_atoms == 8
a = s.lattice.parameters[0]
print(f'a = {a:.4f} A')

CIF: 9008845, space group F -4 3 m, 2 asymmetric sites -> 8 atoms
a = 5.6537 A


## Build a 4x4x4 configuration supercell

`Structure.lattice` is the whole supercell frame (unit cell x 4).

In [3]:
n = (4, 4, 4)
m = s.n_atoms
lat = Lattice.from_parameters(a * n[0], a * n[1], a * n[2], *s.lattice.parameters[3:])
N = m * n[0] * n[1] * n[2]
cell_num = np.arange(N) // m
slot = np.arange(N) % m
cell_index = np.stack(
    [cell_num % n[0], (cell_num // n[0]) % n[1], cell_num // (n[0] * n[1])],
    axis=1,
).astype(np.int64)
supercell = Structure(
    lat,
    s.species[slot],
    (s.frac[slot] + cell_index) / np.array(n, dtype=float),
    model='configuration',
    cell_index=cell_index,
    supercell=n,
    pbc=(True, True, True),
)
print(f'4x4x4 configuration supercell: {supercell.n_atoms} atoms, '
      f'supercell {tuple(int(x) for x in supercell.supercell)}')
assert supercell.n_atoms == 512

4x4x4 configuration supercell: 512 atoms, supercell (4, 4, 4)


## Nearest-neighbour check

In zinc blende every atom has 4 neighbours of the other species at
a*sqrt(3)/4.  The notebook fails loudly if that is wrong.

In [4]:
D = supercell.distance_matrix()
np.fill_diagonal(D, np.inf)
d_min = D.min()
expected = a * np.sqrt(3) / 4
assert abs(d_min - expected) < 1e-6, f'min distance {d_min:.6f} != {expected:.6f}'
nn = (D < 2.6).sum(axis=1)
assert set(nn.tolist()) == {4}, sorted(set(nn.tolist()))
print(f'nearest-neighbour check: min distance {d_min:.6f} A '
      f'(expected {expected:.6f}), 4 neighbours per atom')

nearest-neighbour check: min distance 2.448124 A (expected 2.448124), 4 neighbours per atom


## Convert to ASE and DiffPy

These cells run only if the extras are installed; otherwise they print a
skip message.

In [5]:
try:
    from discuspdf.ase_io import to_ase, from_ase
    atoms = to_ase(s)
    s2 = from_ase(atoms)
    assert s2.isclose(s, rtol=1e-8, atol=1e-12)
    print(f'ASE round trip: OK ({len(atoms)} atoms, dp_* arrays attached)')
except ImportError as e:
    print(f'ASE skipped: {e}')

try:
    from discuspdf.diffpy_io import to_diffpy, from_diffpy
    ds = to_diffpy(s)
    s3 = from_diffpy(ds, model=s.model)
    assert s3.isclose(s, rtol=1e-8, atol=1e-12)
    print(f'DiffPy round trip: OK ({len(ds)} atoms)')
except ImportError as e:
    print(f'DiffPy skipped: {e}')

ASE round trip: OK (8 atoms, dp_* arrays attached)
DiffPy round trip: OK (8 atoms)


## Write a DISCUS .cll and a CIF

In [6]:
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    cll_path = Path(tmpdir) / 'gaas.cll'
    cif_path = Path(tmpdir) / 'gaas.cif'
    from discuspdf.discus_io import write_cll, read_cll
    write_cll(cll_path, s)
    write_cif(cif_path, s)
    s_cll = read_cll(cll_path).structure
    # .cll has 6-decimal precision and regenerates site labels; compare
    # lattice, coordinates, occupancy and B factor (not the exact labels).
    assert np.allclose(s_cll.lattice.parameters, s.lattice.parameters, atol=1e-6)
    assert np.allclose(s_cll.frac, s.frac, atol=1e-6)
    assert np.allclose(s_cll.occupancy, s.occupancy, atol=1e-6)
    # .cll has no B column here, so read_cll defaults B to 1.0; skip the B
    # comparison for this B-free fixture.
    s_cif = read_cif(cif_path).structure
    assert s_cif.isclose(s, rtol=1e-8, atol=1e-9)
    print(f'.cll and CIF written and read back: {s_cll.n_atoms} atoms each')
    print('--- gaas.cll (first 6 lines) ---')
    for line in cll_path.read_text().splitlines()[:6]:
        print(line)

.cll and CIF written and read back: 8 atoms each
--- gaas.cll (first 6 lines) ---
title structure
spcgr P1
cell 5.653700 5.653700 5.653700 90.000000 90.000000 90.000000
atoms
Ga   0.000000 0.000000 0.000000 1.000000
Ga   0.000000 0.500000 0.500000 1.000000


## Summary

* `read_cif` (gemmi) expands the asymmetric unit by the space group.
* `to_ase` / `from_ase` round-trip exactly via `dp_*` arrays.
* `to_diffpy` / `from_diffpy` preserve everything except orientation, site
  labels, atom ids, cell indices, supercell, pbc and the model.
* `write_cif` emits `_discuspdf_*` tags for exact round-trip.